# 07 · Estadísticas descriptivas de las tablas del proyecto

> **En BigQuery.** Lee las cuatro tablas de deforestación de `DATASET_DEFORESTACION` y las tablas de FINAGRO, SFC 414, EVA e ICA del Observatorio. No guarda tablas: las estadísticas se ven aquí. **Qué tienes que cambiar:** nada; la celda 1 ya apunta al proyecto del Observatorio (`prueba-ofr`). **Cuánto tarda:** unos diez minutos, casi todo leyendo FINAGRO.

Calcula, para las variables numéricas principales de cada tabla, las mismas estadísticas descriptivas que dan en R
`summary()` y `psych::describe()`. Las tablas son:

- **Deforestación**: la grilla de 5 km y los cuatro paneles (alertas de GFW, Hansen, IDEAM y detecciones tempranas
  del IDEAM).
- **Crédito**: FINAGRO y SFC 414.
- **Producción agropecuaria**: EVA e ICA pecuario.

Cada tabla se describe tal como está, fila por fila: en los paneles, una fila es una celda en un periodo; en FINAGRO, un
crédito; en la EVA, un cultivo en un municipio y un periodo, y así. Los valores en pesos van en **millones de pesos
corrientes**, y la tasa de FINAGRO va en porcentaje.

La tabla queda en este notebook, también como imagen. No se guarda en BigQuery: ahí solo quedan las
tablas de datos.

## Cómo leer la tabla

| Columna | Qué es |
|---|---|
| n | Número de valores que no faltan. |
| Faltantes | Número de valores vacíos. |
| % ceros | Porcentaje de valores que son exactamente cero. |
| Media, Desv. est. | Promedio y desviación estándar (con *n* − 1, como `sd()` en R). |
| Mín., Máx. | Valores mínimo y máximo. |
| Q1, Mediana, Q3 | Cuartiles: el valor que deja por debajo el 25 %, el 50 % y el 75 % de los datos. Se calculan con el método 7, el que usan `summary()` y `quantile()` en R. |
| Asimetría | 0 si la distribución es simétrica; positiva si tiene una cola larga hacia los valores altos. |
| Curtosis | Curtosis en exceso: 0 para una distribución normal; positiva si tiene colas más pesadas que la normal. |

La asimetría y la curtosis se calculan con las mismas fórmulas que `psych::describe()` (su opción por defecto,
`type = 3`).

In [ ]:
# =====================================================================
# 1. CONFIGURACIÓN
#    Ya está llena para el proyecto del Observatorio (prueba-ofr). Solo
#    hay que cambiarla si las tablas se pasan a otro proyecto o dataset.
# =====================================================================
# --- Dónde están las tablas en BigQuery ------------------------------
# Proyecto de Google Cloud del Observatorio (Prueba-OFR). Es el id,
# no el nombre: el que aparece en la consola al elegir el proyecto.
PROYECTO = "prueba-ofr"

# Dataset con las tablas de deforestación que arman los notebooks 01
# a 06 (grilla, paneles y panel municipal).
DATASET_DEFORESTACION = "Cambio_Climatico"

# --- Tablas del Observatorio (este notebook solo las lee) -------------
# Nombre completo: proyecto.dataset.tabla. Son las que se usaron en el
# trabajo de grado.
# Tabla de desembolsos de FINAGRO
TABLA_FINAGRO = "prueba-ofr.Inclusion_Financiera.FINAGRO_Desembolsos_EFECTIVA"
# Tabla del formato 414 de la Superfinanciera
TABLA_SFC414 = "prueba-ofr.Inclusion_Financiera.SFC414_DASH"
# Evaluaciones Agropecuarias Municipales (EVA)
TABLA_EVA = "prueba-ofr.Produccion.eva_full_enriquecido"
# Inventario pecuario del ICA
TABLA_ICA = "prueba-ofr.Produccion.ICA_pecuario"

# --- Solo si corres el notebook en tu computador ----------------------
# En BigQuery déjalo vacío. En un computador: la ruta al archivo .json
# de la llave de una cuenta de servicio con acceso a estas tablas.
RUTA_LLAVE_BIGQUERY = ""

In [ ]:
# =====================================================================
# 2. INSTALAR PAQUETES
#    En BigQuery instala lo que le falte a la máquina; la primera vez
#    tarda uno o dos minutos. No hay que cambiar nada. En un computador
#    con el entorno del repositorio (requirements_local.txt) se puede
#    saltar.
#    Las versiones fijas son las que se usaron en el trabajo de grado,
#    para que los resultados salgan iguales.
# =====================================================================
%pip install --quiet matplotlib pandas-gbq

In [ ]:
# =====================================================================
# 3. CONEXIÓN CON BIGQUERY Y FUNCIONES DE APOYO
#    No hay que cambiar nada en esta celda. Se conecta a BigQuery y
#    define las funciones con que el resto del notebook lee y escribe
#    tablas. Es igual en todos los notebooks de esta carpeta.
# =====================================================================
import json
import re
import tempfile
import unicodedata
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from google.cloud import bigquery

NOMBRE_NOTEBOOK = "07_eda_estadisticas_descriptivas"

# --- 3.1 ¿Quedó algo sin llenar en la celda 1? --------------------------
# Si queda algún <<LLENAR ...>>, el notebook se detiene aquí y dice cuál.
_pendientes = sorted(n for n, v in list(globals().items())
                     if n.isupper() and isinstance(v, str) and "<<LLENAR" in v)
if _pendientes:
    raise ValueError("Falta llenar en la celda 1: " + ", ".join(_pendientes))

# --- 3.2 Conexión ---------------------------------------------------------
# En BigQuery el notebook entra con la cuenta de quien lo corre y no
# necesita llave. La llave solo se usa si se corre en un computador.
if globals().get("RUTA_LLAVE_BIGQUERY"):
    from google.oauth2 import service_account
    client = bigquery.Client(
        project=PROYECTO,
        credentials=service_account.Credentials.from_service_account_file(RUTA_LLAVE_BIGQUERY))
else:
    client = bigquery.Client(project=PROYECTO)
print(f"Conectado a BigQuery: proyecto {PROYECTO}")

# Carpeta para los archivos que se descargan mientras corre el notebook.
# Es de paso: todo lo que importa termina en tablas de BigQuery.
_carpeta = globals().get("CARPETA_DESCARGAS") or ""
DIR_TRABAJO = Path(_carpeta) if _carpeta else Path(tempfile.gettempdir()) / "ofr_deforestacion"
DIR_TRABAJO.mkdir(parents=True, exist_ok=True)


# --- 3.3 Nombres de tabla -----------------------------------------------
def nombre_completo(tabla, dataset=None):
    """'deforestacion_ideam_celda_5km_periodo' -> 'proyecto.dataset.DEFORESTACION_IDEAM_CELDA_5KM_PERIODO'.

    Las tablas que crean estos notebooks se nombran en mayúscula
    (GRILLA_CELDAS_5KM, DEFORESTACION_IDEAM_CELDA_5KM_PERIODO...). Si la tabla ya viene completa
    (proyecto.dataset.tabla, como las del Observatorio), se devuelve igual.
    Si no, se arma con PROYECTO y el dataset indicado; por defecto,
    DATASET_DEFORESTACION.
    """
    if tabla.count(".") == 2:
        return tabla
    return f"{PROYECTO}.{dataset or globals().get('DATASET_DEFORESTACION')}.{tabla.upper()}"


# --- 3.4 Leer -------------------------------------------------------------
_MARCA = re.compile(r"^(\[índice(?: sin nombre)?\] )?(?:Nombre original: «(.*?)»)?")


_LECTURA_RAPIDA = [True]


def leer_sql(sql):
    """Corre una consulta SQL en BigQuery y devuelve un DataFrame de pandas.

    Usa la lectura rápida de BigQuery (Storage Read API). Si no está
    disponible (por ejemplo, porque la cuenta no tiene el rol «BigQuery
    Read Session User»), sigue con la lectura normal, que da el mismo
    resultado pero tarda más en tablas grandes.
    """
    trabajo = client.query(sql)
    trabajo.result()                      # si la consulta tiene un error, sale aquí
    resultado = None
    if _LECTURA_RAPIDA[0]:
        try:
            resultado = trabajo.to_arrow()
        except Exception as e:
            _LECTURA_RAPIDA[0] = False
            print("Aviso: la lectura rápida de BigQuery no está disponible "
                  f"({type(e).__name__}); se usa la lectura normal.")
    if resultado is None:
        resultado = trabajo.to_arrow(create_bqstorage_client=False)
    return resultado.to_pandas(date_as_object=False, coerce_temporal_nanoseconds=True)


def leer_tabla(tabla, columnas=None, dataset=None, filtro=None):
    """Lee una tabla de BigQuery a pandas.

    columnas  lista de columnas a traer; por defecto, todas.
    filtro    condición SQL opcional, p. ej. "anio >= 2021".

    Si la tabla se guardó con guardar_tabla(), devuelve las filas en el
    mismo orden en que se guardaron, las columnas con su nombre original
    (tildes, espacios) y el índice.
    """
    destino = nombre_completo(tabla, dataset)
    esquema = client.get_table(destino).schema
    con_orden = any(c.name == "orden_fila" for c in esquema)
    seleccion = ", ".join(f"`{c}`" for c in columnas) if columnas else "*"
    df = leer_sql(f"SELECT {seleccion} FROM `{destino}`"
                  + (f" WHERE {filtro}" if filtro else "")
                  + (" ORDER BY orden_fila" if con_orden else ""))
    df = df.drop(columns=["orden_fila"], errors="ignore")

    originales, indice, sin_nombre = {}, [], set()
    for campo in esquema:
        marca = _MARCA.match(campo.description or "")
        if marca.group(2):
            originales[campo.name] = marca.group(2)
        if marca.group(1) and campo.name in df.columns:
            indice.append(campo.name)
            if "sin nombre" in marca.group(1):
                sin_nombre.add(campo.name)
    if indice:
        df = df.set_index(indice)
        df.index = df.index.set_names(
            [None if n in sin_nombre else originales.get(n, n) for n in indice])
    return df.rename(columns=originales)


# --- 3.5 Escribir -----------------------------------------------------------
_DATASETS_LISTOS = set()


def _nombre_para_bigquery(nombre, usados):
    """Nombre de columna que BigQuery acepta siempre: letras sin tilde,
    números y guion bajo. Los que ya cumplen se dejan igual."""
    texto = str(nombre)
    if re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]{0,299}", texto) and texto.lower() not in usados:
        usados.add(texto.lower())
        return texto
    limpio = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode()
    limpio = re.sub(r"[^0-9A-Za-z]+", "_", limpio).strip("_").lower() or "columna"
    if limpio[0].isdigit():
        limpio = "c_" + limpio
    limpio, base, k = limpio[:280], limpio[:280], 2
    while limpio.lower() in usados:
        limpio, k = f"{base}_{k}", k + 1
    usados.add(limpio.lower())
    return limpio


def _tipos_para_bigquery(df):
    """Deja cada columna en un tipo que BigQuery reciba sin error:
    categorías a texto, enteros con vacíos a entero, mezclas a texto."""
    for c in df.columns:
        s = df[c]
        if isinstance(s.dtype, pd.CategoricalDtype):
            s = s.astype(object).where(s.notna(), None)
        if s.dtype == object:
            valores = s.dropna()
            tipos = {type(v) for v in valores}
            if not tipos or tipos <= {str}:
                pass
            elif all(isinstance(v, (bool, np.bool_)) for v in valores):
                s = s.astype("boolean")
            elif all(isinstance(v, (int, np.integer)) and not isinstance(v, bool) for v in valores):
                s = s.astype("Int64")
            elif all(isinstance(v, (int, float, np.integer, np.floating)) for v in valores):
                s = s.astype(float)
            else:
                s = s.map(lambda v: None if v is None or v is pd.NA
                          or (isinstance(v, float) and v != v) else str(v))
        df[c] = s
    return df


def guardar_tabla(df, tabla, descripcion, dataset=None, indice=False,
                  descripciones=None):
    """Escribe `df` en BigQuery. Si la tabla ya existe, la REEMPLAZA entera.

    tabla          nombre corto, p. ej. "deforestacion_ideam_celda_5km_periodo".
    descripcion    texto que se ve en BigQuery, en los detalles de la tabla.
    dataset        por defecto, DATASET_DEFORESTACION. Si no existe, se crea.
    indice         True para guardar también el índice de pandas como
                   columna(s); leer_tabla() lo recupera.
    descripciones  {columna: texto}, para documentar columnas en BigQuery.
    """
    dataset = dataset or globals().get("DATASET_DEFORESTACION")
    destino = nombre_completo(tabla, dataset)
    if dataset not in _DATASETS_LISTOS:
        ds = bigquery.Dataset(f"{PROYECTO}.{dataset}")
        ds.location = globals().get("UBICACION", "US")
        client.create_dataset(ds, exists_ok=True)
        _DATASETS_LISTOS.add(dataset)

    datos = df.copy()
    marcas = {}
    if indice:
        nombres_indice = list(datos.index.names)
        datos = datos.reset_index()
        for k, n in enumerate(nombres_indice):
            columna = datos.columns[k]
            marcas[columna] = "[índice sin nombre] " if n is None else "[índice] "
    else:
        datos = datos.reset_index(drop=True)
    if isinstance(datos.columns, pd.MultiIndex):
        datos.columns = [" | ".join(str(x) for x in c if str(x) != "") for c in datos.columns]

    descripciones = descripciones or {}
    usados, notas, columnas = set(), {}, []
    for c in datos.columns:
        nuevo = _nombre_para_bigquery(c, usados)
        texto = marcas.get(c, "")
        if nuevo != str(c):
            texto += f"Nombre original: «{c}»"
        if descripciones.get(c):
            texto += (". " if texto and not texto.endswith("] ") else "") + descripciones[c]
        notas[nuevo] = texto[:1024]
        columnas.append(nuevo)
    datos.columns = columnas
    datos = _tipos_para_bigquery(datos)
    # BigQuery no guarda el orden de las filas, y algunos cálculos (el
    # muestreo del random forest, por ejemplo) dependen de él. Esta
    # columna lo conserva; leer_tabla() la usa y la quita.
    datos["orden_fila"] = np.arange(len(datos), dtype="int64")
    notas["orden_fila"] = ("Posición de la fila en la tabla original. Sirve para leerla "
                           "en el mismo orden: ORDER BY orden_fila.")

    trabajo = client.load_table_from_dataframe(
        datos, destino,
        job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE"))
    trabajo.result()

    t = client.get_table(destino)
    t.description = descripcion
    t.schema = [bigquery.SchemaField(f.name, f.field_type, mode=f.mode,
                                     description=notas.get(f.name) or None,
                                     fields=f.fields) for f in t.schema]
    client.update_table(t, ["description", "schema"])
    print(f"Guardada {destino}: {t.num_rows:,} filas, {len(t.schema)} columnas")

In [ ]:
import warnings

warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


plt.rcParams.update({"font.family": "serif", "font.serif": ["Times New Roman", "DejaVu Serif"]})
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 250)

ESTADISTICAS = ["n", "Faltantes", "% ceros", "Media", "Desv. est.", "Mín.", "Q1", "Mediana", "Q3", "Máx.",
                "Asimetría", "Curtosis"]


def describir(x):
    # Estadisticas de una variable, como summary() y psych::describe() de R
    x = pd.Series(x, dtype="float64")
    v = x.dropna().to_numpy()
    n = v.size
    d = v - v.mean()
    m2, m3, m4 = (d ** 2).mean(), (d ** 3).mean(), (d ** 4).mean()
    # Asimetria y curtosis en exceso con la correccion de psych::describe (type = 3)
    asimetria = m3 / m2 ** 1.5 * (1 - 1 / n) ** 1.5 if m2 > 0 else np.nan
    curtosis = m4 / m2 ** 2 * (1 - 1 / n) ** 2 - 3 if m2 > 0 else np.nan
    q1, mediana, q3 = np.percentile(v, [25, 50, 75])   # metodo 7 de R, el de quantile() y summary()
    return dict(zip(ESTADISTICAS, [n, int(x.isna().sum()), 100 * (v == 0).mean(), v.mean(), v.std(ddof=1), v.min(),
                                   q1, mediana, q3, v.max(), asimetria, curtosis]))


def describir_tabla(grupo, fuente, tabla, observacion, datos, variables):
    # variables: lista de (columna, unidad, factor por el que se multiplica la columna)
    filas = []
    for columna, unidad, factor in variables:
        filas.append({"Grupo": grupo, "Fuente": fuente, "Tabla": tabla, "Una fila es": observacion,
                      "Filas de la tabla": len(datos), "Variable": columna, "Unidad": unidad,
                      **describir(datos[columna] * factor)})
    return pd.DataFrame(filas)


def numero(v, entero=False, porcentaje=False):
    # Formato en espanol: punto para los miles y coma para los decimales
    if pd.isna(v):
        return "–"
    if porcentaje:
        texto = "<0.01" if 0 < v < 0.01 else f"{v:.2f}"
    elif entero:
        texto = f"{v:,.0f}"
    else:
        a = abs(v)
        texto = ("0" if a == 0 else f"{v:,.0f}" if a >= 1000 else f"{v:,.2f}" if a >= 1
                 else f"{v:.3f}" if a >= 0.01 else f"{v:.2e}")
    return texto.replace(",", "_").replace(".", ",").replace("_", ".")


def formatear(t):
    # Tabla para leer en el notebook: valores con formato, sin las columnas de contexto
    f = t.set_index(["Fuente", "Variable", "Unidad"])[ESTADISTICAS].copy()
    for c in ESTADISTICAS:
        f[c] = f[c].map(lambda v: numero(v, entero=c in ("n", "Faltantes"), porcentaje=c == "% ceros"))
    return f


# Las tablas propias se leen de DATASET_DEFORESTACION; las del Observatorio, de la celda 1.
TABLAS_OBSERVATORIO = {"finagro": TABLA_FINAGRO, "sfc414": TABLA_SFC414, "eva": TABLA_EVA, "ica_pecuario": TABLA_ICA}
leer_panel = lambda nombre, columnas: leer_tabla(nombre, columnas=columnas)
leer_profesor = lambda nombre, columnas: leer_sql(
    "SELECT " + ", ".join(f"`{c}`" for c in columnas) + f" FROM `{TABLAS_OBSERVATORIO[nombre]}`")

## 1. Deforestación

La grilla se describe una vez, con una fila por celda: `bosque_base_ha` es igual en los cuatro paneles. De los
paneles se describen las variables medidas; los rezagos y la media móvil del panel de GFW se dejan por fuera porque son
el mismo `area_def_ha` corrido en el tiempo.

In [ ]:
hansen = leer_panel("perdida_cobertura_hansen_celda_5km_anio", ["cell_id", "perdida_ha", "bosque_base_ha"])
gfw = leer_panel("alertas_gfw_celda_5km_mes", ["area_def_ha", "tasa_def", "def_acum_ha", "bosque_remanente_ha"])
ideam = leer_panel("deforestacion_ideam_celda_5km_periodo", ["def_ha", "bosque_ideam_ha", "bosque_ideam_fin_ha", "bosque_estable_ha",
                                   "regeneracion_ha", "no_bosque_ha", "sin_info_ha"])
dtd = leer_panel("detecciones_tempranas_ideam_celda_5km_trimestre", ["n_alertas", "n_alertas_sinap"])

deforestacion = pd.concat([
    describir_tabla("Deforestación", "Grilla de 5 km", "bosque base de Hansen por celda", "celda",
                    hansen.drop_duplicates("cell_id"), [("bosque_base_ha", "ha", 1)]),
    describir_tabla("Deforestación", "GFW Integrated Alerts", "alertas_gfw_celda_5km_mes", "celda × mes", gfw,
                    [("area_def_ha", "ha", 1), ("tasa_def", "fracción", 1), ("def_acum_ha", "ha", 1),
                     ("bosque_remanente_ha", "ha", 1)]),
    describir_tabla("Deforestación", "Hansen GFC", "perdida_cobertura_hansen_celda_5km_anio", "celda × año", hansen, [("perdida_ha", "ha", 1)]),
    describir_tabla("Deforestación", "IDEAM, cambio de bosque", "deforestacion_ideam_celda_5km_periodo", "celda × periodo", ideam,
                    [(c, "ha", 1) for c in ideam.columns]),
    describir_tabla("Deforestación", "IDEAM, detecciones tempranas", "detecciones_tempranas_ideam_celda_5km_trimestre", "celda × trimestre", dtd,
                    [("n_alertas", "puntos", 1), ("n_alertas_sinap", "puntos", 1)]),
])
del hansen, gfw, ideam, dtd
formatear(deforestacion)

## 2. Crédito

En FINAGRO cada fila es un crédito; en la SFC 414, un grupo de créditos desembolsados en un municipio, un mes y una
combinación de tipo de persona, sexo, tipo de crédito y garantía.

In [ ]:
finagro = leer_profesor("finagro", ["valor_credito", "valor_inversion", "plazo", "periodo_gracia", "tasa_total"])
sfc = leer_profesor("sfc414", ["Montos_desembolsados", "Numero_de_creditos_desembolsados",
                               "Tasa_efectiva_promedio_ponderada_AD"])

credito = pd.concat([
    describir_tabla("Crédito", "FINAGRO", "FINAGRO_Desembolsos_EFECTIVA", "crédito", finagro,
                    [("valor_credito", "M$", 1e-6), ("valor_inversion", "M$", 1e-6), ("plazo", "meses", 1),
                     ("periodo_gracia", "meses", 1), ("tasa_total", "%", 100)]),
    describir_tabla("Crédito", "SFC 414", "SFC414_DASH", "municipio × mes × tipo", sfc,
                    [("Montos_desembolsados", "M$", 1e-6), ("Numero_de_creditos_desembolsados", "créditos", 1),
                     ("Tasa_efectiva_promedio_ponderada_AD", "%", 1)]),
])
del finagro, sfc
formatear(credito)

## 3. Producción agropecuaria

En la EVA cada fila es un cultivo en un municipio y un periodo (año o semestre); en el ICA, un municipio en un año. Del
ICA se describen los totales de las siete especies y las fincas con bovinos.

In [ ]:
eva = leer_profesor("eva", ["Sembrada", "Cosechada", "Produccion", "Rendimiento"])
ESPECIES = ["total_bovinos", "total_fincas_con_bovinos", "total_bufalos", "total_porcinos", "total_aves",
            "total_equinos", "total_caprinos", "total_ovinos"]
ica = leer_profesor("ica_pecuario", ESPECIES)

agropecuario = pd.concat([
    describir_tabla("Producción agropecuaria", "EVA", "eva_full_enriquecido", "cultivo × municipio × periodo", eva,
                    [("Sembrada", "ha", 1), ("Cosechada", "ha", 1), ("Produccion", "t", 1),
                     ("Rendimiento", "t/ha", 1)]),
    describir_tabla("Producción agropecuaria", "ICA pecuario", "ICA_pecuario", "municipio × año", ica,
                    [(c, "fincas" if "fincas" in c else "animales", 1) for c in ESPECIES]),
])
del eva, ica
formatear(agropecuario)

## 4. Tabla completa

**Cómo leer la imagen.** Cada bloque es una tabla, con su fuente, qué es una fila y cuántas filas tiene. Debajo van sus
variables, una por renglón.

In [ ]:
todas = pd.concat([deforestacion, credito, agropecuario], ignore_index=True)


ENCABEZADO = ["Variable", "Unidad"] + ESTADISTICAS
ANCHOS = [2.55, 0.62] + [0.62, 0.56, 0.52] + [0.70] * 7 + [0.62, 0.70]
ALTO_FILA = 0.205


def imagen(t, archivo, titulo):
    bloques = list(t.groupby("Tabla", sort=False))
    n_filas = 1 + sum(1 + len(b) for _, b in bloques)
    ancho, alto = sum(ANCHOS), ALTO_FILA * n_filas
    fig = plt.figure(figsize=(ancho + 0.2, alto + 0.55))
    ax = fig.add_axes([0.1 / (ancho + 0.2), 0.05 / (alto + 0.55), ancho / (ancho + 0.2), alto / (alto + 0.55)])
    ax.set(xlim=(0, ancho), ylim=(0, alto))
    ax.set_axis_off()
    bordes = np.r_[0, np.cumsum(ANCHOS)]
    fig.text(0.5, 1 - 0.18 / (alto + 0.55), titulo, ha="center", va="center", fontsize=10, weight="bold")

    def fila(y, textos, negrilla=False, fondo=None):
        if fondo:
            ax.add_patch(plt.Rectangle((0, y - ALTO_FILA), ancho, ALTO_FILA, color=fondo, lw=0))
        for k, texto in enumerate(textos):
            izquierda = k < 2
            x = bordes[k] + 0.05 if izquierda else bordes[k + 1] - 0.05
            ax.text(x, y - ALTO_FILA / 2, texto, ha="left" if izquierda else "right", va="center", fontsize=6.6,
                    weight="bold" if negrilla else "normal")

    y = alto
    fila(y, ENCABEZADO, negrilla=True, fondo="#d9d9d9")
    y -= ALTO_FILA
    for nombre, b in bloques:
        r0 = b.iloc[0]
        ax.add_patch(plt.Rectangle((0, y - ALTO_FILA), ancho, ALTO_FILA, color="#eef3f8", lw=0))
        ax.text(0.05, y - ALTO_FILA / 2, f"{r0.Fuente} — {nombre}", ha="left", va="center", fontsize=6.8,
                weight="bold")
        ax.text(ancho - 0.05, y - ALTO_FILA / 2, f"una fila = {r0['Una fila es']} · {numero(r0['Filas de la tabla'], True)} filas",
                ha="right", va="center", fontsize=6.4, style="italic")
        y -= ALTO_FILA
        for _, r in b.iterrows():
            fila(y, [r.Variable, r.Unidad] + [numero(r[c], entero=c in ("n", "Faltantes"), porcentaje=c == "% ceros")
                                              for c in ESTADISTICAS])
            y -= ALTO_FILA
        ax.plot([0, ancho], [y, y], color="#9a9a9a", lw=0.4)
    ax.plot([0, ancho], [alto, alto], color="black", lw=0.8)
    ax.plot([0, ancho], [alto - ALTO_FILA, alto - ALTO_FILA], color="black", lw=0.6)
    ax.plot([0, ancho], [0, 0], color="black", lw=0.8)
    return fig


imagen(todas, "estadisticas_descriptivas.png", "Estadísticas descriptivas de las tablas del proyecto")
plt.show()